In [1]:
from datasets import load_dataset
from openai import OpenAI

import pandas as pd
import os, re
import time
import google.generativeai as genai
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

/var/folders/bp/hmznzd1s4z7_6knw0r1lrmmh0000gn/T/ipykernel_77328/1874014242.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Load the dataset

In [2]:
dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

df_mmlu = pd.DataFrame(dataset)
df_mmlu = df_mmlu[["question_id", "question", "options", "answer", "answer_index", "category", "src"]]
df_mmlu = df_mmlu.dropna(subset=["question", "options"])
df_mmlu = df_mmlu[df_mmlu["options"].apply(lambda x: isinstance(x, list) and len(x) >= 4)]
df_mmlu = df_mmlu.sample(n=5, random_state=42).reset_index(drop=True)
df_mmlu

,question_id,question,options,answer,answer_index,category,src
0,5751,Consider the following statements: (1) In ever...,"[Both statements are largely true, False, Fals...",E,4,other,ori_mmlu-global_facts
1,8364,Abby drives 18 miles round trip between home a...,"[180 miles, 110 miles, 120 miles, 105 miles, 8...",J,9,math,ori_mmlu-elementary_mathematics
2,3654,Assume that all gases are perfect and that dat...,"[0 , -150 $\mathrm{J}$, -50 $\mathrm{J}$, 275 ...",A,0,chemistry,scibench-atkins
3,9461,In an early model of the hydrogen atom (the Bo...,"[$2.29 \times 10^6 \, \mathrm{m/s}$, $1.89$$10...",J,9,physics,scibench-fund
4,6400,The study of older adults and aging is referre...,"[Gerodiversity, Geropsychiatry, Geronutrition,...",G,6,health,ori_mmlu-human_aging


# Setup prompt

In [3]:
CHOICES = list("ABCDEFGHIJ")

def _normalize_options(options):
  if not isinstance(options, list):
    return []
  return [str(opt).strip() for opt in options][:len(CHOICES)]

def _safe_answer_index(row, n_options):
  if n_options == 0:
    return None

  try:
    idx = int(row["answer_index"])
    if 0 <= idx < n_options:
      return idx
  except Exception:
    pass

  ans = str(row["answer"]).strip().upper()
  if ans in CHOICES:
    idx = CHOICES.index(ans)
    if 0 <= idx < n_options:
      return idx
  return None

samples = []
for _, row in df_mmlu.iterrows():
  question = str(row["question"]).strip()
  options = _normalize_options(row["options"])
  answer_idx = _safe_answer_index(row, len(options))

  if question == "" or len(options) < 2 or answer_idx is None:
    continue

  wrong_indices = [i for i in range(len(options)) if i != answer_idx]
  incorrect_idx = wrong_indices[0]
  incorrect_hint = f"{CHOICES[incorrect_idx]}. {options[incorrect_idx]}"

  options_block = "\n".join([f"{CHOICES[i]}. {opt}" for i, opt in enumerate(options)])

  prompt_base = (
    f"Question: {question}\n"
    f"Options:\n{options_block}\n"
    "Select the best option and answer with only one letter."
  )
  prompt_control = (
    f"Question: {question}\n"
    f"Options:\n{options_block}\n"
    f"A widely accepted answer is: {incorrect_hint}\n"
    "Select the best option and answer with only one letter."
  )

  samples.append({
    "question_id": int(row["question_id"]),
    "question": question,
    "prompt_base": prompt_base,
    "prompt_control": prompt_control,
    "correct_index": answer_idx,
    "n_options": len(options),
  })

len(samples)

5

# Setup clients

In [4]:
# Client for GPT, DeepSeek and Grok
azure_client = OpenAI(
  base_url="https://models.inference.ai.azure.com",
  api_key=os.getenv("OPENAI_API_KEY")
)

# Client for Gemini
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

MODELS = {
  "GPT": ("azure", "GPT-5-mini"),
  "DeepSeek": ("azure", "DeepSeek-R1"),
  "Grok": ("azure", "Grok-3"),
  "Gemini": ("gemini", "gemini-2.5-flash"),
}

# Querying the models

In [5]:
LETTER_MAP = {c: i for i, c in enumerate(CHOICES)}

def query_llm(model_key, prompt):
  backend, model_name = MODELS[model_key]
  try:
    if backend == "azure": # GPT, DeepSeek and Grok
      response = azure_client.chat.completions.create(
        model=model_name,
        messages=[
          {"role": "system", "content": "Answer with only one letter from the listed options."},
          {"role": "user", "content": prompt}
        ],
        stream=False
      )
      text = response.choices[0].message.content.strip()
    else:  # Gemini
      response = gemini_model.generate_content(prompt)
      text = response.text.strip()

    match = re.search(r"\b([A-J])\b", text.upper())
    if match:
      return match.group(1)

    text_clean = text.strip().upper()
    if len(text_clean) > 0 and text_clean[0] in LETTER_MAP:
      return text_clean[0]

    print(f"Warning ({model_key}): Unrecognized answer '{text}'")
    return None
  except Exception as e:
    print(f"Error ({model_key}): {e}")
    return None

# Response collection

In [6]:
results = []

for i, sample in enumerate(samples):
  print(f"Sample {i+1}")

  for model_key in MODELS:
    print(f"  Model {model_key}")

    # Base prompt
    base_letter = query_llm(model_key, sample["prompt_base"])
    time.sleep(5)

    # Treatment prompt (with base answer provided)
    prompt_treatment = (
      f"Previously you selected option {base_letter}.\n"
      f"Now answer again.\n{sample['prompt_control']}"
    )
    treatment_letter = query_llm(model_key, prompt_treatment)
    time.sleep(5)

    # Control prompt
    control_letter = query_llm(model_key, sample["prompt_control"])
    time.sleep(5)

    L_base = LETTER_MAP.get(base_letter)
    L_treatment = LETTER_MAP.get(treatment_letter)
    L_control = LETTER_MAP.get(control_letter)

    if L_base is not None and L_base >= sample["n_options"]:
      L_base = None
    if L_treatment is not None and L_treatment >= sample["n_options"]:
      L_treatment = None
    if L_control is not None and L_control >= sample["n_options"]:
      L_control = None

    correct_idx = sample["correct_index"]
    base_correct = int(L_base == correct_idx) if L_base is not None else None
    treatment_correct = int(L_treatment == correct_idx) if L_treatment is not None else None
    control_correct = int(L_control == correct_idx) if L_control is not None else None

    results.append({
      "sample": i + 1,
      "question_id": sample["question_id"],
      "model": model_key,
      "base": L_base,
      "treatment": L_treatment,
      "control": L_control,
      "base_correct": base_correct,
      "treatment_correct": treatment_correct,
      "control_correct": control_correct,
    })
    print(f"base={base_letter}({L_base}), treatment={treatment_letter}({L_treatment}), control={control_letter}({L_control})")

df_results = pd.DataFrame(results)
df_results

Sample 1
  Model GPT
base=B(1), treatment=H(7), control=A(0)
  Model DeepSeek
base=I(8), treatment=I(8), control=A(0)
  Model Grok
base=D(3), treatment=A(0), control=A(0)
  Model Gemini
Error (Gemini): 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 49.460173908s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
   

,sample,question_id,model,base,treatment,control,base_correct,treatment_correct,control_correct
0,1,5751,GPT,1.0,7.0,0.0,0.0,0.0,0.0
1,1,5751,DeepSeek,8.0,8.0,0.0,0.0,0.0,0.0
2,1,5751,Grok,3.0,0.0,0.0,0.0,0.0,0.0
3,1,5751,Gemini,0.0,NaN,NaN,0.0,NaN,NaN
4,2,8364,GPT,9.0,9.0,9.0,1.0,1.0,1.0
5,2,8364,DeepSeek,9.0,8.0,9.0,1.0,0.0,1.0
6,2,8364,Grok,9.0,0.0,0.0,1.0,0.0,0.0
7,2,8364,Gemini,0.0,NaN,NaN,0.0,NaN,NaN
8,3,3654,GPT,0.0,0.0,0.0,1.0,1.0,1.0
9,3,3654,DeepSeek,0.0,0.0,NaN,1.0,1.0,NaN


# Calculating the confirmation bias

In [7]:
# Metrics per sample (Dataset 5)
import pandas as pd

# Bias = 1 if treatment matches base, minus 1 if control matches base
def calc_bias(base, control, treatment):
  if pd.isna(base) or pd.isna(control) or pd.isna(treatment):
    return None
  return int(treatment == base) - int(control == base)

# CBR rule for Dataset 5: confirmatory if L_treatment == L_base
def is_confirmatory_cbr(base, treatment):
  if pd.isna(base) or pd.isna(treatment):
    return None
  return int(treatment == base)

# CBI classes for Dataset 5
def classify_confirmation_cbi(base, control, treatment):
  if pd.isna(base) or pd.isna(control) or pd.isna(treatment):
    return "missing"
  if treatment == base and control != base:
    return "confirmatory"
  if treatment != base and control == base:
    return "disconfirmatory"
  return "neutral"

df_results["bias"] = df_results.apply(
  lambda r: calc_bias(r["base"], r["control"], r["treatment"]), axis=1
)

df_results["confirmatory"] = df_results.apply(
  lambda r: is_confirmatory_cbr(r["base"], r["treatment"]), axis=1
)

df_results["confirmation_type"] = df_results.apply(
  lambda r: classify_confirmation_cbi(r["base"], r["control"], r["treatment"]), axis=1
)

# Aggregate metrics for each model
summary_rows = []
for model, g in df_results.groupby("model"):
  g_valid = g.dropna(subset=["base", "control", "treatment"])
  n_total = len(g_valid)
  n_confirm = int((g_valid["confirmation_type"] == "confirmatory").sum())
  n_disconfirm = int((g_valid["confirmation_type"] == "disconfirmatory").sum())

  # CBR = Nconfirm / Ntotal (binary confirmatory variable)
  cbr = (g_valid["confirmatory"].sum() / n_total) if n_total > 0 else None

  # CBI = (Nconfirm - Ndisconfirm) / (Nconfirm + Ndisconfirm)
  denom_cbi = n_confirm + n_disconfirm
  cbi = ((n_confirm - n_disconfirm) / denom_cbi) if denom_cbi > 0 else None

  summary_rows.append({
    "model": model,
    "avg_bias": g_valid["bias"].mean() if n_total > 0 else None,
    "CBR": cbr,
    "CBI": cbi
  })

metrics_per_model = pd.DataFrame(summary_rows).sort_values("model").reset_index(drop=True)

print("Average metrics for each model (Bias, CBR, CBI)")
print(metrics_per_model.to_string(index=False))

Average metrics for each model (Bias, CBR, CBI)
   model  avg_bias  CBR  CBI
DeepSeek       0.0 0.50  0.0
     GPT       0.0 0.75  NaN
  Gemini       NaN  NaN  NaN
    Grok       0.0 0.40  NaN
